# CRISP-DM Phase 6 – Deployment

**Goal:** put the model in the clinician's hands: a web module where a febrile patient is registered and the result shows the
probability of each arbovirus, the suggested diagnosis, **why**, and the next steps.

Previous phase: `05_evaluation.ipynb`

## 6.1 Saved model

`python -m src.train` saves `models/model.pkl` with the model (retrained on all 2023 data), the class names, the dengue alert
threshold, the feature labels and every metric; `models/metrics.json` is the readable version without the model.

## 6.2 Using the model as the web app does

`src/prediction.py` loads the model **once** and, for each patient, returns the probabilities, the suggestion and the
SHAP explanation (XGBoost and LightGBM compute it natively, so the server does not need the `shap` package).

In [1]:
import sys
sys.path.insert(0, "..")  # project root, to import the src package

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)

from src.prediction import NEXT_STEPS, Predictor
from src.preprocessing import CLASS_LABELS

predictor = Predictor()
same = {"fever": True, "arthralgia": True, "myalgia": True, "headache": True}
patients = {
    "Adult, fever + joint pain + arthritis, Minas Gerais, March": dict(age=45, sex="F", days_since_onset=2,
        findings={**same, "arthritis": True}, state="31", month=3),
    "Young adult, fever + retro-orbital pain + leukopenia, São Paulo, April": dict(age=24, sex="M", days_since_onset=3,
        findings={"fever": True, "headache": True, "retro_orbital_pain": True, "myalgia": True, "leukopenia": True},
        state="35", month=4),
    "SAME symptoms in Minas Gerais (MG), March": dict(age=35, sex="F", days_since_onset=2, findings=same, state="31", month=3),
    "SAME symptoms in Paraná (PR), March": dict(age=35, sex="F", days_since_onset=2, findings=same, state="41", month=3),
}
for name, patient in patients.items():
    out = predictor.predict(**patient)
    probs = ", ".join(f"{CLASS_LABELS[c]} {p:.0%}" for c, p in out["probabilities"].items())
    print(f"\n{name}\n  -> {CLASS_LABELS[out['prediction']]} (alert: {out['dengue_alert']}) | {probs}")
    for e in out["explanation"][:4]:
        print(f"     {e['label']:<38} {e['shap']:+.2f}")


Adult, fever + joint pain + arthritis, Minas Gerais, March
  -> Chikungunya (alert: False) | Dengue 2%, Chikungunya 76%, Other febrile illness 21%
     State of residence                     +0.89
     Joint pain                             +0.30
     Joint swelling (arthritis)             +0.15
     Days since symptom onset               -0.11

Young adult, fever + retro-orbital pain + leukopenia, São Paulo, April
  -> Dengue (alert: True) | Dengue 97%, Chikungunya 1%, Other febrile illness 2%
     State of residence                     +1.29
     Leukopenia (low white blood cells)     +0.59
     Joint pain                             +0.32
     Days since symptom onset               +0.25

SAME symptoms in Minas Gerais (MG), March
  -> Chikungunya (alert: False) | Dengue 6%, Chikungunya 68%, Other febrile illness 26%
     State of residence                     +0.87
     Joint pain                             +0.38
     Days since symptom onset               -0.13
     Age          

## 6.3 Web application

| Component | File | Role |
|---|---|---|
| Routes | `app.py` | `/register` consultation · `/result/<id>` · `/patients` · `/about` · `/methodology` |
| Prediction | `src/prediction.py` | Probabilities, dengue alert, SHAP explanation, next steps |
| Explanations | `src/explain.py` | TreeSHAP values per patient |
| Validation | `src/validation.py` | Server-side validation of the form |
| Database | `src/database.py` | `consultations` table; findings, probabilities and explanation stored as JSON |
| Hosting | `render.yaml`, `requirements.txt` | Render with gunicorn; library versions pinned to the training ones |

Run locally:

```bash
venv\Scripts\activate
flask run --debug          # http://127.0.0.1:5000
```

## 6.4 Monitoring and maintenance

- **Retrain yearly:** `python -m src.sinan --years <train> <validation>` then `python -m src.train`.
- **Watch for drift:** compare the share of each suggested diagnosis with the circulation reported by surveillance.
- **Close the loop:** store the laboratory result of each consultation, so the hospital builds its own labeled dataset with
  numeric lab values for the next iteration.
- **Keep the notice:** clinical decision support only.